In [1]:
"""
Cross-Validation Stability Check — Radius-Based 21-Feature Set
===============================================================
Validates two candidate models on the radius-based feature set:
  A. Seed sweep — 20 different random 80/20 train/test splits
  B. Repeated stratified K-fold — 5 folds x 5 repeats = 25 estimates
"""
 
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import f1_score, recall_score, precision_score, confusion_matrix
import warnings
warnings.filterwarnings("ignore")

In [2]:
# Config
 
TARGET_COL   = "diagnosis"
TARGET_POS   = "M"
DROP_COLS    = ["id", "Unnamed: 32"]
N_SEEDS      = 30
N_SPLITS_CV  = 10
N_REPEATS_CV = 10
CV_SEED      = 42
 

RADIUS_BASED_DROP = [
    "area_mean",    "area_se",    "area_worst",
    "perimeter_mean", "perimeter_se", "perimeter_worst",
    "concavity_mean", "concavity_se", "concavity_worst",
]


In [3]:
# Data loading
 
def load_radius_based_data():
    df = pd.read_csv("Data for Task 1.csv")
    df = df.drop(columns=[c for c in DROP_COLS if c in df.columns])
    y_str = df[TARGET_COL]
    X = df.drop(columns=[TARGET_COL]).drop(columns=RADIUS_BASED_DROP)
    print(f"Radius-based feature set: {len(X.columns)} features")
    print(f"  {list(X.columns)}\n")
    return X, y_str

In [4]:
def evaluate_split(model, X_train, X_test, y_train, y_test, threshold):
    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc = scaler.transform(X_test)

    model.fit(X_train_sc, y_train)

    y_prob = model.predict_proba(X_test_sc)[:, 1]
    y_pred = np.where(y_prob >= threshold, TARGET_POS, "B")

    cm = confusion_matrix(y_test, y_pred, labels=["B", "M"])

    return {
        "f1": f1_score(y_test, y_pred, pos_label="M"),
        "recall": recall_score(y_test, y_pred, pos_label="M"),
        "precision": precision_score(y_test, y_pred, pos_label="M"),
        "fn": cm[1, 0],
        "fp": cm[0, 1]
    }

In [11]:
models = {
    "Logistic Regression C=1.0 th=0.45": {
        "model": LogisticRegression(
            C=1.0,
            solver="lbfgs",
            max_iter=10000,
            class_weight="balanced"
        ),
        "threshold": 0.45
    },
    
    "SVM C=0.01 th=0.45": {
        "model": SVC(
            kernel="linear",
            C=0.01,
            probability=True,
            class_weight="balanced"
        ),
        "threshold": 0.45
    }
}

In [12]:
# Stability check A: seed sweep
 
X, y = load_radius_based_data()

seed_results = []

for name, cfg in models.items():
    for seed in range(N_SEEDS):

        X_train, X_test, y_train, y_test = train_test_split(
            X,
            y,
            test_size=0.20,
            stratify=y,
            random_state=seed
        )

        result = evaluate_split(
            cfg["model"],
            X_train,
            X_test,
            y_train,
            y_test,
            cfg["threshold"]
        )

        result["model"] = name
        result["seed"] = seed

        seed_results.append(result)

seed_df = pd.DataFrame(seed_results)

seed_summary = seed_df.groupby("model").agg(
    mean_f1=("f1", "mean"),
    std_f1=("f1", "std"),
    mean_recall=("recall", "mean"),
    mean_fn=("fn", "mean"),
    max_fn=("fn", "max"),
    fn_ge_2=("fn", lambda x: (x >= 2).sum()),
    mean_fp=("fp", "mean")
)

seed_summary["pct_fn_ge_2"] = (
    100 * seed_summary["fn_ge_2"] / N_SEEDS
).round(1)

print(seed_summary.round(4))

Radius-based feature set: 21 features
  ['radius_mean', 'texture_mean', 'smoothness_mean', 'compactness_mean', 'concave points_mean', 'symmetry_mean', 'fractal_dimension_mean', 'radius_se', 'texture_se', 'smoothness_se', 'compactness_se', 'concave points_se', 'symmetry_se', 'fractal_dimension_se', 'radius_worst', 'texture_worst', 'smoothness_worst', 'compactness_worst', 'concave points_worst', 'symmetry_worst', 'fractal_dimension_worst']

                                   mean_f1  std_f1  mean_recall  mean_fn  \
model                                                                      
Logistic Regression C=1.0 th=0.45   0.9539  0.0207       0.9643      1.5   
SVM C=0.01 th=0.45                  0.9508  0.0216       0.9500      2.1   

                                   max_fn  fn_ge_2  mean_fp  pct_fn_ge_2  
model                                                                     
Logistic Regression C=1.0 th=0.45       3       14   2.4333         46.7  
SVM C=0.01 th=0.45         

In [13]:
# Stability check B: repeated stratified K-fold
 
rkf = RepeatedStratifiedKFold(
    n_splits=N_SPLITS_CV,
    n_repeats=N_REPEATS_CV,
    random_state=CV_SEED
)

cv_results = []

for name, cfg in models.items():

    for fold, (train_idx, test_idx) in enumerate(rkf.split(X, y), start=1):

        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]

        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]

        result = evaluate_split(
            cfg["model"],
            X_train,
            X_test,
            y_train,
            y_test,
            cfg["threshold"]
        )

        result["model"] = name
        result["fold"] = fold

        cv_results.append(result)

cv_df = pd.DataFrame(cv_results)

print(cv_df.groupby("model").agg(
    mean_f1=("f1", "mean"),
    std_f1=("f1", "std"),
    mean_recall=("recall", "mean"),
    mean_fn=("fn", "mean"),
    max_fn=("fn", "max"),
    fn_ge_2=("fn", lambda x: (x >= 2).sum()),
    mean_fp=("fp", "mean")
))

                                    mean_f1    std_f1  mean_recall  mean_fn  \
model                                                                         
Logistic Regression C=1.0 th=0.45  0.958539  0.030192     0.969286     0.65   
SVM C=0.01 th=0.45                 0.958091  0.028009     0.958896     0.87   

                                   max_fn  fn_ge_2  mean_fp  
model                                                        
Logistic Regression C=1.0 th=0.45       3       13     1.14  
SVM C=0.01 th=0.45                      4       20     0.90  
